In [48]:
import sqlite3 as sql
import pandas as pd

In [49]:
connect = sql.connect('games.db') #conecta ao arquivo do banco

df = pd.read_csv('Video_Games.csv') #carrega o dataset 

df.head()#mostra 5 primeiros

,index,Name,Platform,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating
0,0,Wii Sports,Wii,2006.0,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8,322.0,Nintendo,E
1,1,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,NaN,NaN,NaN,NaN,NaN,NaN
2,2,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,Nintendo,E
3,3,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8,192.0,Nintendo,E
4,4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,NaN,NaN,NaN,NaN,NaN,NaN


In [50]:
df['User_Score'] = pd.to_numeric(df['User_Score'], errors = 'coerce') #substitui coluna atual pela versão tratada

print(f"Valores vazios no User Score:", df['User_Score'].isnull().sum()) # confere o total de valores vazios na coluna

Valores vazios no User Score: 9210


In [51]:
with open ('schema.sql', 'r', encoding = 'utf-8') as arquive: #abre o arquivo em modo leitura 'r = read'
    command_sql = arquive.read()

cursor = connect.cursor() #cria quem envia comandos ao banco

cursor.executescript(command_sql) #executa o sql e salva alterações
connect.commit()

print(f'Tabelas criadas com sucesso!')

Tabelas criadas com sucesso!


In [52]:
publisher_unique = df['Publisher'].dropna().unique() #extrai os nomes únicos, remove valores NaN

df_publisher = pd.DataFrame ({ # nova tabela para publishers
    'publisher_id': range(1, len(publisher_unique) +1),
    'name': publisher_unique
})

print(f'Total de publisher únicas: {len(df_publisher)}')
df_publisher.head()

Total de publisher únicas: 581


,publisher_id,name
0,1,Nintendo
1,2,Microsoft Game Studios
2,3,Take-Two Interactive
3,4,Sony Computer Entertainment
4,5,Activision


In [53]:
df_publisher.to_sql('publisher', connect, if_exists = 'append', index = False) #traduz a tabela do pandas para banco de dados

print(f'As 581 publishers foram inseridas no banco com sucesso!')


As 581 publishers foram inseridas no banco com sucesso!


In [54]:
print(f'Jogos sem Nome (Name):', df['Name'].isnull().sum()) #verificando dados 
print(f'Jogos sem Publisher (Publisher):', df['Publisher'].isnull().sum())

Jogos sem Nome (Name): 2
Jogos sem Publisher (Publisher): 55


In [55]:
df_clean = df.dropna(subset = ['Name', 'Publisher']).copy() #joga fora os jogos quer vieram sem Nome ou Publisher

df_cross = df_clean.merge(df_publisher, left_on ='Publisher', right_on = 'name') #junta tabela original com a tabela feita acima DataFrame

df_game = df_cross[['Name', 'Year_of_Release', 'Platform', 'Genre', 'publisher_id']].copy() #recorta apenas as colunas que vão para o banco

df_game.columns = ['name', 'year', 'platform', 'genre', 'publisher_id'] #renomeia de acordo com o schema.sql

df_game.insert(0, 'game_id', range(1, len (df_game) +1)) #cria o game_id

print(f'Total de jogos preparados para o banco: {len(df_game)}')
df_game.head()

Total de jogos preparados para o banco: 16871


,game_id,name,year,platform,genre,publisher_id
0,1,Wii Sports,2006.0,Wii,Sports,1
1,2,Super Mario Bros.,1985.0,NES,Platform,1
2,3,Mario Kart Wii,2008.0,Wii,Racing,1
3,4,Wii Sports Resort,2009.0,Wii,Sports,1
4,5,Pokemon Red/Pokemon Blue,1996.0,GB,Role-Playing,1


In [56]:
df_game['year'] = df_game['year'].astype('Int64') #converte para Inteiro flexível

df_game.to_sql('game', connect, if_exists = 'append', index = False) #envia tabela para o sql

print(f'A tabela Game foi inserida no banco com sucesso!')

A tabela Game foi inserida no banco com sucesso!


In [57]:
df_value = df_cross[['Global_Sales', 'Critic_Score', 'User_Score', 'Rating']].copy()

df_value.columns = ['globals', 'critics', 'users', 'rating']

df_value.insert(0, 'game_id', df_game['game_id']) #FK

df_value.insert(0, 'val_id', range(1, len(df_value) +1))

df_value.to_sql('value', connect, if_exists = 'append', index = False)

print(f'Valores de value para o banco: {len(df_value)}')
df_value.head()

Valores de value para o banco: 16871


,val_id,game_id,globals,critics,users,rating
0,1,1,82.53,76.0,8.0,E
1,2,2,40.24,NaN,NaN,NaN
2,3,3,35.52,82.0,8.3,E
3,4,4,32.77,80.0,8.0,E
4,5,5,31.37,NaN,NaN,NaN
